In [41]:
import pennylane as qml
from pennylane import numpy as pnp

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [42]:
iris = load_iris()

X = iris.data
y = iris.target

# scale features to [0, π] for angle encoding
scaler = MinMaxScaler(feature_range=(0, pnp.pi))
X = scaler.fit_transform(X)

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [43]:
n_qubits = 4
n_layers = 3

dev = qml.device("default.qubit", wires=n_qubits)

In [56]:
@qml.qnode(dev)
def circuit(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    
    # Use qml.math.stack to preserve gradient tracking across the list
    return qml.math.stack([qml.expval(qml.PauliZ(i)) for i in range(3)])

In [57]:
def model(x, weights):
    return circuit(x, weights)

In [58]:
def one_hot(y):
    # If y is 1, pnp.eye(3)[1] returns [0., 1., 0.]
    return pnp.eye(3)[y]

In [63]:
def cost(weights):
    # logits shape initially: (3, 120)
    logits = circuit(X_train, weights) 
    
    # Transpose logits to shape: (120, 3)
    logits = qml.math.T(logits)
    
    # y_true shape: (120, 3)
    y_true = pnp.eye(3)[y_train]
    
    # Now they line up perfectly!
    return pnp.mean((logits - y_true) ** 2)

In [64]:
weights = pnp.random.normal(
    0, 0.1,
    (n_layers, n_qubits, 3),
    requires_grad=True
)

In [65]:
opt = qml.AdamOptimizer(stepsize=0.05)

In [66]:
epochs = 50
loss_history = []

for epoch in range(epochs):

    weights = opt.step(cost, weights)

    current_loss = cost(weights)
    loss_history.append(current_loss)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | Loss = {current_loss:.4f}")

Epoch 00 | Loss = 0.3422
Epoch 05 | Loss = 0.1847
Epoch 10 | Loss = 0.1227
Epoch 15 | Loss = 0.1145
Epoch 20 | Loss = 0.1026
Epoch 25 | Loss = 0.0885
Epoch 30 | Loss = 0.0809
Epoch 35 | Loss = 0.0747
Epoch 40 | Loss = 0.0687
Epoch 45 | Loss = 0.0653


In [67]:
# 1. Get predictions on the test set
# Remember to transpose the test logits just like we did in the cost function
test_logits = qml.math.T(circuit(X_test, weights))

# 2. Get the index of the highest expectation value as the predicted class
predictions = pnp.argmax(test_logits, axis=1)

# 3. Calculate accuracy
accuracy = pnp.mean(predictions == y_test)
print(f"Final Test Accuracy: {accuracy * 100:.2f}%")

Final Test Accuracy: 93.33%
